## Exercise 1

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 10: Transfer Learning
# Niveau: Experten
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. Synthetischer Few-Shot Datensatz ───────────────────────────────────────
np.random.seed(42)

N_KLASSEN       = 10   # 10 Klassen
K_SCHUSS        = 5    # 5 Beispiele pro Klasse für Prototypen
N_TEST_PRO_KL   = 20   # Testbeispiele pro Klasse

EINGABE_DIM     = 64
EMBEDDING_DIM   = 32

# Jede Klasse hat ein "Zentrum" im Eingaberaum
klassen_zentren = np.random.randn(N_KLASSEN, EINGABE_DIM).astype("float32")

def beispiele_erstellen(n_kl, k, eingang_dim, zentren, rauschen=0.3):
    """Erstellt k Beispiele pro Klasse aus normalverteiltem Rauschen um Zentrum."""
    X, y = [], []
    for kl in range(n_kl):
        for _ in range(k):
            x = zentren[kl] + np.random.randn(eingang_dim) * rauschen
            X.append(x.astype("float32"))
            y.append(kl)
    return np.array(X), np.array(y)

# Support-Set (K Beispiele pro Klasse)
X_support, y_support = beispiele_erstellen(
    N_KLASSEN, K_SCHUSS, EINGABE_DIM, klassen_zentren, rauschen=0.3
)

# Abfrage-Set (Testbeispiele)
X_abfrage, y_abfrage = beispiele_erstellen(
    N_KLASSEN, N_TEST_PRO_KL, EINGABE_DIM, klassen_zentren, rauschen=0.3
)

print(f"Support-Set:  {X_support.shape} ({K_SCHUSS} Beispiele × {N_KLASSEN} Klassen)")
print(f"Abfrage-Set: {X_abfrage.shape} ({N_TEST_PRO_KL} Beispiele × {N_KLASSEN} Klassen)")

# ── 2. Embedding-Netzwerk trainieren ──────────────────────────────────────────
# Zunächst das Embedding auf einem größeren "Pre-Training" Datensatz trainieren
X_pretrain, y_pretrain = beispiele_erstellen(
    N_KLASSEN, 100, EINGABE_DIM, klassen_zentren, rauschen=0.5
)

embedding_netz = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation="relu", input_shape=(EINGABE_DIM,)),
    tf.keras.layers.Dense(64,  activation="relu"),
    tf.keras.layers.Dense(EMBEDDING_DIM, name="embedding"),  # L2-normalisiert
], name="Embedding_Netz")

# Vortraining mit Klassifikation (dann Embedding nutzen)
modell_pretrain = tf.keras.Sequential([
    embedding_netz,
    tf.keras.layers.Dense(N_KLASSEN, activation="softmax"),
])
modell_pretrain.compile(optimizer="adam",
                         loss="sparse_categorical_crossentropy",
                         metrics=["accuracy"])

print("\nVortraining des Embeddings...")
modell_pretrain.fit(X_pretrain, y_pretrain, epochs=20,
                    batch_size=64, verbose=0)
print("Vortraining abgeschlossen.")

# ── 3. Prototypen berechnen ───────────────────────────────────────────────────
def prototypen_berechnen(embedding_modell, X_support, y_support, n_klassen):
    """
    Berechnet Klassen-Prototypen als Mittelwert der Embeddings pro Klasse.
    """
    embeddings = embedding_modell.predict(X_support, verbose=0)
    prototypen = np.zeros((n_klassen, EMBEDDING_DIM), dtype="float32")
    for kl in range(n_klassen):
        maske = y_support == kl
        prototypen[kl] = embeddings[maske].mean(axis=0)
    return prototypen

prototypen = prototypen_berechnen(
    embedding_netz, X_support, y_support, N_KLASSEN
)
print(f"\nPrototypen berechnet: {prototypen.shape}")

# ── 4. Klassifikation durch nächsten Prototyp ─────────────────────────────────
def naechster_prototyp_klassifizieren(embedding_modell, X_test, prototypen):
    """
    Klassifiziert Testbeispiele durch den nächstgelegenen Prototyp
    (euklidische Distanz im Embedding-Raum).
    """
    test_embeddings = embedding_modell.predict(X_test, verbose=0)
    vorhersagen     = []

    for emb in test_embeddings:
        # Euklidische Distanz zu allen Prototypen
        distanzen  = np.linalg.norm(prototypen - emb, axis=1)
        naechste   = np.argmin(distanzen)
        vorhersagen.append(naechste)

    return np.array(vorhersagen)

vorhersagen = naechster_prototyp_klassifizieren(
    embedding_netz, X_abfrage, prototypen
)

genauigkeit = np.mean(vorhersagen == y_abfrage)
print(f"\nFew-Shot Genauigkeit ({K_SCHUSS}-Shot, {N_KLASSEN}-Way): {genauigkeit:.4f}")

# ── 5. Vergleich mit einfachem Nearest Neighbor (Roh-Features, kein Embedding) ─
from sklearn.neighbors import KNeighborsClassifier
knn_basis = KNeighborsClassifier(n_neighbors=1)
knn_basis.fit(X_support, y_support)
knn_pred  = knn_basis.predict(X_abfrage)
knn_acc   = np.mean(knn_pred == y_abfrage)
print(f"1-NN auf Roh-Features (Baseline):  {knn_acc:.4f}")
print(f"Prototypische Methode (mit Emb.): {genauigkeit:.4f}")

# ── 6. Konfusionsmatrix visualisieren ─────────────────────────────────────────
confusion_matrix = np.zeros((N_KLASSEN, N_KLASSEN), dtype="int32")
for wahr, pred in zip(y_abfrage, vorhersagen):
    confusion_matrix[wahr, pred] += 1

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im = axes[0].imshow(confusion_matrix, cmap="Blues")
axes[0].set_title(f"Konfusionsmatrix (Few-Shot, {K_SCHUSS}-Shot)\n"
                   f"Genauigkeit: {genauigkeit:.4f}")
axes[0].set_xlabel("Vorhergesagt")
axes[0].set_ylabel("Tatsächlich")
plt.colorbar(im, ax=axes[0])

# Embedding-Raum visualisieren (t-SNE Ersatz: 2D Projektion)
from sklearn.decomposition import PCA
alle_X = np.vstack([X_abfrage, X_support])
alle_y = np.hstack([y_abfrage, y_support])
alle_emb = embedding_netz.predict(alle_X, verbose=0)
pca_2d  = PCA(n_components=2).fit_transform(alle_emb)

farben = plt.cm.tab10(np.linspace(0, 1, N_KLASSEN))
n_test = len(X_abfrage)

for kl in range(N_KLASSEN):
    maske_test   = (np.arange(n_test) < n_test) & (alle_y[:n_test] == kl)
    maske_support = (np.arange(len(alle_y)) >= n_test) & (alle_y == kl)

    axes[1].scatter(pca_2d[:n_test][alle_y[:n_test]==kl, 0],
                     pca_2d[:n_test][alle_y[:n_test]==kl, 1],
                     color=farben[kl], alpha=0.5, s=20, label=f"Kl.{kl}")
    idx_sup = np.where(maske_support)[0]
    axes[1].scatter(pca_2d[idx_sup, 0], pca_2d[idx_sup, 1],
                     color=farben[kl], s=100, marker="*", edgecolors="black")

# Prototypen im PCA-Raum
pca_proto = PCA(n_components=2).fit(alle_emb).transform(prototypen)
axes[1].scatter(pca_proto[:, 0], pca_proto[:, 1],
                 s=200, marker="D", c=range(N_KLASSEN), cmap="tab10",
                 edgecolors="black", zorder=10, label="Prototypen")
axes[1].set_title("Embedding-Raum (PCA 2D)\n★=Support, ◆=Prototyp")
axes[1].legend(fontsize=6, ncol=2)
axes[1].grid(True, alpha=0.3)

plt.suptitle(f"Prototypisches Few-Shot Learning: {K_SCHUSS}-Shot {N_KLASSEN}-Way",
             fontsize=13)
plt.tight_layout()
plt.savefig("E10_1_few_shot.png", dpi=100)
plt.show()
print("Diagramm gespeichert: E10_1_few_shot.png")


## Exercise 2

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 10: Transfer Learning
# Niveau: Experten
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. MNIST-Daten ────────────────────────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32").reshape(-1, 784) / 255.0
x_test  = x_test.astype("float32").reshape(-1, 784)  / 255.0
N_KLASSEN = 10

# ── 2. Großes Lehrer-Modell ───────────────────────────────────────────────────
def lehrer_modell_erstellen():
    return tf.keras.Sequential([
        tf.keras.layers.Dense(1024, activation="relu", input_shape=(784,)),
        tf.keras.layers.Dense(512,  activation="relu"),
        tf.keras.layers.Dense(256,  activation="relu"),
        tf.keras.layers.Dense(N_KLASSEN),  # Logits (kein Softmax)
    ], name="Lehrer")

# ── 3. Kleines Schüler-Modell ─────────────────────────────────────────────────
def schueler_modell_erstellen():
    return tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation="relu", input_shape=(784,)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(N_KLASSEN),  # Logits (kein Softmax)
    ], name="Schüler")

# ── 4. Lehrer trainieren ──────────────────────────────────────────────────────
print("Trainiere Lehrer-Modell...")
lehrer = lehrer_modell_erstellen()
lehrer.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)
lehrer.fit(x_train, y_train, epochs=10, batch_size=128,
            validation_split=0.1, verbose=0)

lehrer_acc = lehrer.evaluate(x_test, y_test, verbose=0)[1]
print(f"Lehrer Test-Genauigkeit: {lehrer_acc:.4f}")
print(f"Lehrer Parameter:        {lehrer.count_params():,}")

# ── 5. Schüler ohne Destillation (Baseline) ───────────────────────────────────
print("\nTrainiere Schüler OHNE Destillation (Baseline)...")
schueler_basis = schueler_modell_erstellen()
schueler_basis.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)
h_basis = schueler_basis.fit(
    x_train, y_train, epochs=15, batch_size=128,
    validation_split=0.1, verbose=0
)
basis_acc = schueler_basis.evaluate(x_test, y_test, verbose=0)[1]
print(f"Schüler (Baseline) Test-Acc: {basis_acc:.4f}")

# ── 6. Knowledge Distillation Verlustfunktion ────────────────────────────────
TEMPERATUR  = 3.0   # Hohe Temperatur → weichere Verteilungen
ALPHA       = 0.7   # Gewichtung: 0.7 × Destillationsverlust + 0.3 × Hard-Loss

def destillations_verlust(y_wahr, lehrer_logits, schueler_logits, temperatur, alpha):
    """
    L = α * L_KD + (1-α) * L_CE
    L_KD: KL-Divergenz zwischen Lehrer- und Schüler-Softmaxes (mit Temperatur)
    L_CE: Standard Cross-Entropy mit wahren Labels
    """
    # Weiche Ausgaben: Softmax mit Temperatur
    lehrer_weich  = tf.nn.softmax(lehrer_logits  / temperatur, axis=-1)
    schueler_weich = tf.nn.softmax(schueler_logits / temperatur, axis=-1)

    # KL-Divergenz (destilliertes Wissen)
    kl_loss = tf.keras.losses.KLDivergence()(
        lehrer_weich, schueler_weich
    ) * (temperatur ** 2)

    # Harte Labels (normale Cross-Entropy)
    ce_loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)(
        y_wahr, schueler_logits
    )

    return alpha * kl_loss + (1.0 - alpha) * ce_loss

# ── 7. Schüler MIT Knowledge Distillation trainieren ─────────────────────────
print("\nTrainiere Schüler MIT Knowledge Distillation...")
schueler_kd = schueler_modell_erstellen()
optimizer   = tf.keras.optimizers.Adam(0.001)

EPOCHEN    = 15
BATCH_GR   = 128

kd_verlauf  = []
kd_val_accs = []

# Lehrer-Vorhersagen (alle Logits vorausberechnen für Geschwindigkeit)
lehrer_logits_all = lehrer.predict(x_train, batch_size=256, verbose=0)
lehrer_logits_val = lehrer.predict(x_train[int(0.9*len(x_train)):], verbose=0)

# Übliche 90/10 Split
n_val = int(0.1 * len(x_train))
x_tr_kd   = x_train[:-n_val]
y_tr_kd   = y_train[:-n_val]
l_tr_kd   = lehrer_logits_all[:-n_val]

x_val_kd  = x_train[-n_val:]
y_val_kd  = y_train[-n_val:]

for epoche in range(EPOCHEN):
    # Shuffling
    perm = np.random.permutation(len(x_tr_kd))
    x_sh, y_sh, l_sh = x_tr_kd[perm], y_tr_kd[perm], l_tr_kd[perm]

    epoch_losses = []
    for start in range(0, len(x_sh), BATCH_GR):
        x_b = x_sh[start:start+BATCH_GR]
        y_b = y_sh[start:start+BATCH_GR]
        l_b = l_sh[start:start+BATCH_GR]

        with tf.GradientTape() as tape:
            schueler_logits = schueler_kd(x_b, training=True)
            verlust = destillations_verlust(
                y_b, tf.cast(l_b, tf.float32),
                schueler_logits, TEMPERATUR, ALPHA
            )
        grads = tape.gradient(verlust, schueler_kd.trainable_variables)
        optimizer.apply_gradients(zip(grads, schueler_kd.trainable_variables))
        epoch_losses.append(float(verlust))

    # Validierung
    val_logits  = schueler_kd(x_val_kd, training=False)
    val_pred    = np.argmax(val_logits.numpy(), axis=1)
    val_acc     = np.mean(val_pred == y_val_kd)
    kd_verlauf.append(np.mean(epoch_losses))
    kd_val_accs.append(val_acc)
    print(f"Epoche {epoche+1:2d}/{EPOCHEN} | Loss: {np.mean(epoch_losses):.4f} | "
          f"Val-Acc: {val_acc:.4f}")

kd_acc = np.mean(
    np.argmax(schueler_kd.predict(x_test, verbose=0), axis=1) == y_test
)
print(f"\nSchüler (KD) Test-Acc: {kd_acc:.4f}")
print(f"Schüler (Basis) Test-Acc: {basis_acc:.4f}")
print(f"Verbesserung durch Destillation: {(kd_acc - basis_acc)*100:+.2f}%")
print(f"Lehrer Test-Acc: {lehrer_acc:.4f} | Parameter: {lehrer.count_params():,}")
print(f"Schüler Parameter: {schueler_kd.count_params():,} "
      f"({schueler_kd.count_params()/lehrer.count_params()*100:.1f}% des Lehrers)")

# ── 8. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(h_basis.history["val_accuracy"], label="Schüler (Baseline)", linewidth=2)
axes[0].plot(kd_val_accs,                      label="Schüler (KD)", linewidth=2)
axes[0].axhline(lehrer_acc, color="gray", linestyle=":",
                 label=f"Lehrer ({lehrer_acc:.4f})")
axes[0].set_title("Validierungs-Genauigkeit: KD vs. Baseline")
axes[0].set_xlabel("Epoche")
axes[0].legend()
axes[0].grid(True)

methoden = ["Lehrer\n(groß)", "Schüler\n(Baseline)", "Schüler\n(KD)"]
accs     = [lehrer_acc, basis_acc, kd_acc]
params   = [lehrer.count_params(), schueler_basis.count_params(), schueler_kd.count_params()]
farben_b = ["#e74c3c", "#3498db", "#2ecc71"]

axes[1].bar(methoden, accs, color=farben_b, edgecolor="black")
for i, (a, p) in enumerate(zip(accs, params)):
    axes[1].text(i, a + 0.005, f"{a:.4f}\n{p/1000:.0f}K Params",
                  ha="center", fontsize=9)
axes[1].set_title("Test-Genauigkeit und Modellgröße")
axes[1].set_ylabel("Test-Genauigkeit")
axes[1].set_ylim(0, 1.1)
axes[1].grid(True, axis="y", alpha=0.3)

plt.suptitle(f"Knowledge Distillation: Lehrer→Schüler (T={TEMPERATUR}, α={ALPHA})",
             fontsize=13)
plt.tight_layout()
plt.savefig("E10_2_knowledge_distillation.png", dpi=100)
plt.show()
print("Diagramm gespeichert: E10_2_knowledge_distillation.png")


## Exercise 3

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 10: Transfer Learning
# Niveau: Experten
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. MNIST-Daten ────────────────────────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32").reshape(-1, 784) / 255.0
x_test  = x_test.astype("float32").reshape(-1, 784)  / 255.0

# ── 2. Modell trainieren ──────────────────────────────────────────────────────
modell = tf.keras.Sequential([
    tf.keras.layers.Dense(512, activation="relu", input_shape=(784,)),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10,  activation="softmax"),
], name="Pruning_Modell")

modell.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Trainiere Modell (unverdichtet)...")
modell.fit(x_train, y_train, epochs=10, batch_size=128,
           validation_split=0.1, verbose=0)

basis_acc = modell.evaluate(x_test, y_test, verbose=0)[1]
print(f"Basis-Genauigkeit (keine Pruning): {basis_acc:.4f}")
print(f"Basis-Parameter:                   {modell.count_params():,}")

# ── 3. Magnituden-basiertes Pruning ──────────────────────────────────────────
def gewichte_beschneiden(modell, schwellwert):
    """
    Beschneidet Gewichte mit |w| < Schwellwert → setzt sie auf 0.
    Gibt die Sparsität (Anteil Nullgewichte) zurück.
    """
    gesamt = 0
    null_gesamt = 0
    for schicht in modell.layers:
        gewichte = schicht.get_weights()
        neue_gewichte = []
        for w in gewichte:
            maske = np.abs(w) >= schwellwert
            w_beschnitten = w * maske
            neue_gewichte.append(w_beschnitten)
            gesamt      += w.size
            null_gesamt += np.sum(~maske)
        schicht.set_weights(neue_gewichte)
    return null_gesamt / gesamt if gesamt > 0 else 0.0

def sparsitaet_messen(modell):
    """Misst den Anteil der Nullgewichte im Modell."""
    gesamt     = 0
    null_anz   = 0
    for schicht in modell.layers:
        for w in schicht.get_weights():
            gesamt   += w.size
            null_anz += np.sum(w == 0)
    return null_anz / gesamt if gesamt > 0 else 0.0

# ── 4. Verschiedene Schwellwerte testen ───────────────────────────────────────
schwellwerte = [0.0, 0.01, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5]
ergebnisse   = []

print("\nPruning-Analyse (verschiedene Schwellwerte):")
print(f"{'Schwellwert':>12} {'Sparsität':>12} {'Test-Acc':>12}")
print("-" * 38)

for schwellwert in schwellwerte:
    # Modell frisch laden (Pruning ist destruktiv → Modell kopieren)
    modell_kopie = tf.keras.models.clone_model(modell)
    modell_kopie.set_weights(modell.get_weights())
    modell_kopie.compile(optimizer="adam",
                          loss="sparse_categorical_crossentropy",
                          metrics=["accuracy"])

    # Pruning anwenden
    sparsitaet = gewichte_beschneiden(modell_kopie, schwellwert)

    # Genauigkeit nach Pruning
    acc = modell_kopie.evaluate(x_test, y_test, verbose=0)[1]

    ergebnisse.append({
        "Schwellwert": schwellwert,
        "Sparsität":   sparsitaet,
        "Acc":         acc,
    })
    print(f"{schwellwert:>12.3f} {sparsitaet:>11.2%} {acc:>12.4f}")

# ── 5. Pruning + Retraining ───────────────────────────────────────────────────
print("\n── Pruning mit anschließendem Retraining ──")
# Moderat beschnittenes Modell (z.B. Schwellwert=0.1) retrainieren
modell_retrain = tf.keras.models.clone_model(modell)
modell_retrain.set_weights(modell.get_weights())

# Pruning
spars_vor_rt = gewichte_beschneiden(modell_retrain, schwellwert=0.1)
acc_vor_rt   = modell.evaluate(x_test, y_test, verbose=0)[1]
print(f"Vor Retraining: Sparsität={spars_vor_rt:.2%}, Acc≈unverd.")

# Retraining (wenige Epochen mit kleiner LR)
modell_retrain.compile(
    optimizer=tf.keras.optimizers.Adam(lr=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
modell_retrain.fit(x_train, y_train, epochs=5, batch_size=128,
                    validation_split=0.1, verbose=0)

acc_nach_rt = modell_retrain.evaluate(x_test, y_test, verbose=0)[1]
spars_nach_rt = sparsitaet_messen(modell_retrain)

print(f"Nach Retraining: Sparsität={spars_nach_rt:.2%}, Acc={acc_nach_rt:.4f}")

# ── 6. Visualisierung ─────────────────────────────────────────────────────────
sparsitaeten = [r["Sparsität"]   for r in ergebnisse]
accs_pruning = [r["Acc"]         for r in ergebnisse]
schwellw_lis = [r["Schwellwert"] for r in ergebnisse]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Genauigkeit vs. Sparsität
axes[0].plot([s*100 for s in sparsitaeten], accs_pruning,
             marker="o", linewidth=2, color="steelblue")
axes[0].axhline(basis_acc, color="red", linestyle="--",
                 label=f"Basis ({basis_acc:.4f})")
axes[0].set_title("Test-Genauigkeit vs. Sparsität")
axes[0].set_xlabel("Sparsität (%)")
axes[0].set_ylabel("Test-Genauigkeit")
axes[0].legend()
axes[0].grid(True)
axes[0].set_ylim(0, 1.05)

# Genauigkeit vs. Schwellwert
axes[1].plot(schwellw_lis, accs_pruning, marker="s", linewidth=2, color="green")
axes[1].axhline(basis_acc, color="red", linestyle="--", label="Basis")
axes[1].set_title("Test-Genauigkeit vs. Pruning-Schwellwert")
axes[1].set_xlabel("Pruning-Schwellwert")
axes[1].set_ylabel("Test-Genauigkeit")
axes[1].legend()
axes[1].grid(True)

# Gewichtsverteilung vor/nach Pruning
w_vor_pruning = np.concatenate([
    schicht.get_weights()[0].flatten()
    for schicht in modell.layers
    if len(schicht.get_weights()) > 0
])
w_nach_pruning = np.concatenate([
    schicht.get_weights()[0].flatten()
    for schicht in modell_retrain.layers
    if len(schicht.get_weights()) > 0
])

# Nur Nicht-Null-Gewichte für Visualisierung
w_vor_nz  = w_vor_pruning[w_vor_pruning  != 0]
w_nach_nz = w_nach_pruning[w_nach_pruning != 0]

axes[2].hist(w_vor_nz,  bins=80, alpha=0.6, color="#3498db",
             label="Vor Pruning",  density=True)
axes[2].hist(w_nach_nz, bins=80, alpha=0.6, color="#e74c3c",
             label="Nach Pruning+Retrain", density=True)
axes[2].axvline(0, color="black", linewidth=1)
axes[2].set_title("Gewichtsverteilung (Nicht-Null-Werte)")
axes[2].set_xlabel("Gewichtswert")
axes[2].set_ylabel("Dichte")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle("Magnitudenbasiertes Pruning: Genauigkeit vs. Sparsität", fontsize=13)
plt.tight_layout()
plt.savefig("E10_3_pruning.png", dpi=100)
plt.show()
print("Diagramm gespeichert: E10_3_pruning.png")
print("\n── Zusammenfassung ──")
print(f"Keine Pruning:        {basis_acc:.4f}  | Sparsität: 0.00%")
dreizig_idx = next(i for i, r in enumerate(ergebnisse) if r["Sparsität"] >= 0.3)
print(f"Pruning@30%:          {ergebnisse[dreizig_idx]['Acc']:.4f}  | Sparsität: {ergebnisse[dreizig_idx]['Sparsität']:.2%}")
print(f"Pruning+Retrain@10%: {acc_nach_rt:.4f}  | Sparsität: {spars_nach_rt:.2%}")
